## Load Data + Basic Exploration

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [2]:
df_transactions = pd.read_csv('../data/transactions_data.csv')
df_users = pd.read_csv('../data/users_data.csv')
df_cards = pd.read_csv('../data/cards_data.csv')
df_mcc_codes = pd.read_csv('../data/mcc_codes.csv', delimiter=';')
df_fraud_labels = pd.read_csv('../data/binary_train_labels.csv')

In [3]:
def column_summary(df):
    """
    Helper function to summarize dataframe columns.

    Args:
        df (pandas.DataFrame)

    Returns:
        summary_df (pandas.DataFrame)
    """
    summary_data = []
    for col_name in df.columns:
        col_dtype = df[col_name].dtype
        num_of_nulls = df[col_name].isnull().sum()
        num_of_non_nulls = df[col_name].notnull().sum()
        num_of_distinct_values = df[col_name].nunique()

        if num_of_distinct_values <= 10:
            distinct_values_counts = df[col_name].value_counts().to_dict()
        else:
            top_10_values_counts = df[col_name].value_counts().head(10).to_dict()
            distinct_values_counts = {k: v for k, v in sorted(top_10_values_counts.items(), key= lambda item: item[1], reverse=True )}
        summary_data.append({
            'col_name': col_name,
            'col_dtype': col_dtype,
            'num_of_nulls': num_of_nulls,
            'num_of_non_nulls': num_of_non_nulls,
            'num_of_distinct_values': num_of_distinct_values,
            'distinct_values_counts': distinct_values_counts
        })
    summary_df = pd.DataFrame(summary_data)
    return summary_df

#### Transactions data

In [4]:
df_transactions.head(5)

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [5]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 12 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              int64  
 1   date            object 
 2   client_id       int64  
 3   card_id         int64  
 4   amount          object 
 5   use_chip        object 
 6   merchant_id     int64  
 7   merchant_city   object 
 8   merchant_state  object 
 9   zip             float64
 10  mcc             int64  
 11  errors          object 
dtypes: float64(1), int64(5), object(6)
memory usage: 1.2+ GB


In [6]:
df_transactions.describe()

,id,client_id,card_id,merchant_id,zip,mcc
count,1.330592e+07,1.330592e+07,1.330592e+07,1.330592e+07,1.165321e+07,1.330592e+07
mean,1.558402e+07,1.026812e+03,3.475268e+03,4.772376e+04,5.132782e+04,5.565440e+03
std,4.704499e+06,5.816386e+02,1.674356e+03,2.581534e+04,2.940423e+04,8.757002e+02
min,7.475327e+06,0.000000e+00,0.000000e+00,1.000000e+00,1.001000e+03,1.711000e+03
25%,1.150604e+07,5.190000e+02,2.413000e+03,2.588700e+04,2.860200e+04,5.300000e+03
50%,1.557087e+07,1.070000e+03,3.584000e+03,4.592600e+04,4.767000e+04,5.499000e+03
75%,1.965361e+07,1.531000e+03,4.901000e+03,6.757000e+04,7.790100e+04,5.812000e+03
max,2.376187e+07,1.998000e+03,6.144000e+03,1.003420e+05,9.992800e+04,9.402000e+03


In [7]:
summary_transactions = column_summary(df_transactions)
display(summary_transactions)  

,col_name,col_dtype,num_of_nulls,num_of_non_nulls,num_of_distinct_values,distinct_values_counts
0,id,int64,0,13305915,13305915,"{7475327: 1, 7475328: 1, 7475329: 1, 7475331: ..."
1,date,object,0,13305915,4136496,"{'2016-03-03 11:42:00': 18, '2011-06-09 12:46:..."
2,client_id,int64,0,13305915,1219,"{1098: 48479, 909: 43381, 1963: 42462, 1776: 4..."
3,card_id,int64,0,13305915,4071,"{4938: 31552, 2408: 30672, 3239: 30520, 3233: ..."
4,amount,object,0,13305915,81161,"{'$80.00': 132115, '$100.00': 128867, '$60.00'..."
5,use_chip,object,0,13305915,3,"{'Swipe Transaction': 6967185, 'Chip Transacti..."
6,merchant_id,int64,0,13305915,74831,"{59935: 610053, 27092: 589140, 61195: 562410, ..."
7,merchant_city,object,0,13305915,12492,"{'ONLINE': 1563700, 'Houston': 146917, 'Miami'..."
8,merchant_state,object,1563700,11742215,199,"{'CA': 1427087, 'TX': 1010207, 'NY': 857510, '..."
9,zip,float64,1652706,11653209,25256,"{98516.0: 36753, 91606.0: 31337, 87121.0: 3067..."


#### Users data

In [8]:
df_users.head(5)

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [9]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 2000 non-null   int64  
 1   current_age        2000 non-null   int64  
 2   retirement_age     2000 non-null   int64  
 3   birth_year         2000 non-null   int64  
 4   birth_month        2000 non-null   int64  
 5   gender             2000 non-null   object 
 6   address            2000 non-null   object 
 7   latitude           2000 non-null   float64
 8   longitude          2000 non-null   float64
 9   per_capita_income  2000 non-null   object 
 10  yearly_income      2000 non-null   object 
 11  total_debt         2000 non-null   object 
 12  credit_score       2000 non-null   int64  
 13  num_credit_cards   2000 non-null   int64  
dtypes: float64(2), int64(7), object(5)
memory usage: 218.9+ KB


In [10]:
df_users.describe()

,id,current_age,retirement_age,birth_year,birth_month,latitude,longitude,credit_score,num_credit_cards
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,999.500000,45.391500,66.237500,1973.803000,6.439000,37.389225,-91.554765,709.734500,3.073000
std,577.494589,18.414092,3.628867,18.421234,3.565338,5.114324,16.283293,67.221949,1.637379
min,0.000000,18.000000,50.000000,1918.000000,1.000000,20.880000,-159.410000,480.000000,1.000000
25%,499.750000,30.000000,65.000000,1961.000000,3.000000,33.837500,-97.395000,681.000000,2.000000
50%,999.500000,44.000000,66.000000,1975.000000,7.000000,38.250000,-86.440000,711.500000,3.000000
75%,1499.250000,58.000000,68.000000,1989.000000,10.000000,41.200000,-80.130000,753.000000,4.000000
max,1999.000000,101.000000,79.000000,2002.000000,12.000000,61.200000,-68.670000,850.000000,9.000000


In [11]:
summary_users = column_summary(df_users)
display(summary_users)

,col_name,col_dtype,num_of_nulls,num_of_non_nulls,num_of_distinct_values,distinct_values_counts
0,id,int64,0,2000,2000,"{825: 1, 1746: 1, 1718: 1, 708: 1, 1164: 1, 68..."
1,current_age,int64,0,2000,80,"{18: 77, 47: 47, 49: 44, 22: 43, 50: 43, 28: 4..."
2,retirement_age,int64,0,2000,29,"{65: 314, 66: 295, 67: 259, 68: 195, 69: 162, ..."
3,birth_year,int64,0,2000,80,"{1970: 53, 1998: 47, 1972: 43, 2002: 42, 1991:..."
4,birth_month,int64,0,2000,12,"{2: 197, 1: 192, 11: 189, 8: 171, 3: 167, 12: ..."
5,gender,object,0,2000,2,"{'Female': 1016, 'Male': 984}"
6,address,object,0,2000,1999,"{'506 Washington Lane': 2, '766 Third Drive': ..."
7,latitude,float64,0,2000,989,"{29.76: 22, 41.83: 19, 40.71: 15, 25.77: 13, 4..."
8,longitude,float64,0,2000,1224,"{-95.38: 24, -87.68: 19, -80.13: 13, -73.94: 1..."
9,per_capita_income,object,0,2000,1754,"{'$0': 15, '$19382': 4, '$21869': 3, '$22599':..."


#### Cards data

In [12]:
df_cards.head(5)

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [13]:
df_cards.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6146 entries, 0 to 6145
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id                     6146 non-null   int64 
 1   client_id              6146 non-null   int64 
 2   card_brand             6146 non-null   object
 3   card_type              6146 non-null   object
 4   card_number            6146 non-null   int64 
 5   expires                6146 non-null   object
 6   cvv                    6146 non-null   int64 
 7   has_chip               6146 non-null   object
 8   num_cards_issued       6146 non-null   int64 
 9   credit_limit           6146 non-null   object
 10  acct_open_date         6146 non-null   object
 11  year_pin_last_changed  6146 non-null   int64 
 12  card_on_dark_web       6146 non-null   object
dtypes: int64(6), object(7)
memory usage: 624.3+ KB


In [14]:
df_cards.describe()

,id,client_id,card_number,cvv,num_cards_issued,year_pin_last_changed
count,6146.000000,6146.000000,6.146000e+03,6146.000000,6146.000000,6146.000000
mean,3072.500000,994.939636,4.820426e+15,506.220794,1.503091,2013.436707
std,1774.341709,578.614626,1.328582e+15,289.431123,0.519191,4.270699
min,0.000000,0.000000,3.001055e+14,0.000000,1.000000,2002.000000
25%,1536.250000,492.250000,4.486365e+15,257.000000,1.000000,2010.000000
50%,3072.500000,992.000000,5.108957e+15,516.500000,1.000000,2013.000000
75%,4608.750000,1495.000000,5.585237e+15,756.000000,2.000000,2017.000000
max,6145.000000,1999.000000,6.997197e+15,999.000000,3.000000,2020.000000


In [15]:
summary_cards = column_summary(df_cards)
display(summary_cards)

,col_name,col_dtype,num_of_nulls,num_of_non_nulls,num_of_distinct_values,distinct_values_counts
0,id,int64,0,6146,6146,"{4524: 1, 2731: 1, 3701: 1, 42: 1, 4659: 1, 45..."
1,client_id,int64,0,6146,2000,"{1741: 9, 797: 9, 1301: 9, 20: 8, 989: 8, 1416..."
2,card_brand,object,0,6146,4,"{'Mastercard': 3209, 'Visa': 2326, 'Amex': 402..."
3,card_type,object,0,6146,3,"{'Debit': 3511, 'Credit': 2057, 'Debit (Prepai..."
4,card_number,int64,0,6146,6146,"{4344676511950444: 1, 4956965974959986: 1, 458..."
5,expires,object,0,6146,259,"{'02/2020': 377, '01/2020': 130, '01/2021': 93..."
6,cvv,int64,0,6146,998,"{740: 15, 877: 15, 939: 14, 269: 13, 32: 13, 4..."
7,has_chip,object,0,6146,2,"{'YES': 5500, 'NO': 646}"
8,num_cards_issued,int64,0,6146,3,"{1: 3114, 2: 2972, 3: 60}"
9,credit_limit,object,0,6146,3654,"{'$0': 31, '$8700': 25, '$8000': 25, '$9300': ..."


#### MCC decoder

In [16]:
df_mcc_codes.head(5)

,mcc_code,description
0,5812,Eating Places and Restaurants
1,5541,Service Stations
2,7996,"Amusement Parks, Carnivals, Circuses"
3,5411,"Grocery Stores, Supermarkets"
4,4784,Tolls and Bridge Fees


In [17]:
df_mcc_codes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   mcc_code     109 non-null    int64 
 1   description  109 non-null    object
dtypes: int64(1), object(1)
memory usage: 1.8+ KB


In [18]:
summary_mcc_codes = column_summary(df_mcc_codes)
display(summary_mcc_codes)

,col_name,col_dtype,num_of_nulls,num_of_non_nulls,num_of_distinct_values,distinct_values_counts
0,mcc_code,int64,0,109,109,"{5812: 1, 5541: 1, 7996: 1, 5411: 1, 4784: 1, ..."
1,description,object,0,109,108,"{'Passenger Railways': 2, 'Eating Places and R..."


#### Fraud labels

In [19]:
df_fraud_labels.head(5)

,Unnamed: 0,target
0,10000000,0
1,10000002,0
2,10000003,0
3,10000004,0
4,10000005,0


In [20]:
df_fraud_labels.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8914963 entries, 0 to 8914962
Data columns (total 2 columns):
 #   Column      Dtype
---  ------      -----
 0   Unnamed: 0  int64
 1   target      int64
dtypes: int64(2)
memory usage: 136.0 MB


In [21]:
df_fraud_labels.describe()

,Unnamed: 0,target
count,8.914963e+06,8.914963e+06
mean,1.558473e+07,1.495463e-03
std,4.703991e+06,3.864230e-02
min,7.475327e+06,0.000000e+00
25%,1.150786e+07,0.000000e+00
50%,1.557140e+07,0.000000e+00
75%,1.965387e+07,0.000000e+00
max,2.376187e+07,1.000000e+00


In [22]:
summary_fraud_labels = column_summary(df_fraud_labels)
display(summary_fraud_labels)

,col_name,col_dtype,num_of_nulls,num_of_non_nulls,num_of_distinct_values,distinct_values_counts
0,Unnamed: 0,int64,0,8914963,8914963,"{10000000: 1, 10000002: 1, 10000003: 1, 100000..."
1,target,int64,0,8914963,2,"{0: 8901631, 1: 13332}"


## Data Cleaning

#### 1) Create a SINGLE dataset

In [23]:
df = pd.merge(df_transactions, df_users, how='left', left_on='client_id', right_on='id', suffixes=("_trans", "_cli"))

In [24]:
df.head()

,id_trans,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,id_cli,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN,1556,30,67,1989,7,Female,594 Mountain View Street,46.80,-100.76,$23679,$48277,$110153,740,4
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN,561,48,67,1971,6,Male,604 Pine Street,40.80,-91.12,$18076,$36853,$112139,834,5
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN,1129,49,65,1970,4,Male,2379 Forest Lane,33.18,-117.29,$16894,$34449,$36540,686,3
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN,430,52,67,1967,5,Female,903 Hill Boulevard,41.42,-87.35,$26168,$53350,$128676,685,5
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN,848,51,69,1968,5,Male,166 River Drive,38.86,-76.60,$33529,$68362,$96182,711,2


In [25]:
df = pd.merge(df, df_cards, how='left', left_on='card_id', right_on='id')

In [26]:
sum(~df['client_id_x']==df['client_id_y']) # If this is 0, it means that the client_id in 'cards_data' matches the client_id of the main df, as one expects

0

In [27]:
# Create a dictionary from the MCC codes dataframe
mcc_dict = df_mcc_codes.set_index('mcc_code')['description'].to_dict()

df['merchant_category'] = df['mcc'].apply(lambda x: mcc_dict[x])

In [28]:
df = pd.merge(df, df_fraud_labels, how='left', left_on='id_trans', right_on = 'Unnamed: 0')

#### 2) Convert to appropiate data types

In [29]:
# Date fields

df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d %H:%M:%S')
df['expires'] = pd.to_datetime(df['expires'], format='%m/%Y')
df['acct_open_date'] = pd.to_datetime(df['acct_open_date'], format='%m/%Y')

# Quant fields

df['amount'] = df['amount'].str.replace('$', '').astype(float) # The str accessor lets us apply string operators vectorized accross entire Series
df['per_capita_income'] = df['per_capita_income'].str.replace('$', '').astype(float)
df['yearly_income'] = df['yearly_income'].str.replace('$', '').astype(float)
df['total_debt'] = df['total_debt'].str.replace('$', '').astype(float)
df['credit_limit'] = df['credit_limit'].str.replace('$', '').astype(float)

# Target -> binary

df['target'] = df['target'].astype('Int64')

In [30]:
df.dtypes

id_trans                          int64
date                     datetime64[ns]
client_id_x                       int64
card_id                           int64
amount                          float64
use_chip                         object
merchant_id                       int64
merchant_city                    object
merchant_state                   object
zip                             float64
mcc                               int64
errors                           object
id_cli                            int64
current_age                       int64
retirement_age                    int64
birth_year                        int64
birth_month                       int64
gender                           object
address                          object
latitude                        float64
longitude                       float64
per_capita_income               float64
yearly_income                   float64
total_debt                      float64
credit_score                      int64


#### 3) Remove rare values & fill NULLS

Some transactions have **negative amounts**. This can mean different things, but most likely it indicates refunds. Since it is not clear from the context how to interpret this, we will verify the ratio of transactions with negative amounts, and if it's low enough, we will drop them from the dataset.

In [31]:
print(f"Ratio of transactions with negative amounts: {len(df[df['amount']<0])/len(df):.2f}.")
print(f"Ratio of fraudulent transactions with negative amounts: {len(df[(df['target'] == 1) & (df['amount']<0)])/len(df[df['target']==1]):.2f}.")

Ratio of transactions with negative amounts: 0.05.
Ratio of fraudulent transactions with negative amounts: 0.04.


In [32]:
# Since it's less then 5% in both cases, we can drop them.

df = df[df['amount']>=0]
print(f'Number of entries: {len(df)}.')

Number of entries: 12645866.


Next, we need to fill null values. Luckily, only the columns ['merchant_state', 'zip', 'errors', 'Unnamed: 0', 'target'] have null values. The last two columns have null values because some transactions were left unlabeled for the Hackathon. We will use some of these unlabeled transactions to test our final, deployed solution. But for the modeling phase, we will drop the unlabeled transactions.

In [33]:
df = df[~df['target'].isnull()]
print(f'Number of entries: {len(df)}.')

Number of entries: 8472184.


The ZIP and merchant_state have null values because of the ONLINE transactions. Since we already have the field 'merchant_city', we will simply drop these two fields. As for the 'errors' column, we can assume that if it's null, it means that no errors happened. We can therefore fill the nulls with "No errors".

In [34]:
df['errors'] = df['errors'].fillna('No errors')

#### 4) Aggregate features

Several features that would be available in production can be computed from this dataset:

1) Transaction count per user
2) Average transaction amount per user
3) Maximum transaction amount per user
4) Card activity (transaction count per card)

All aggregations are computed from **legitimate transactions only** (target=0).

*Note: As mentioned in the Readme file, this is a clear case of data leakage. We are using the whole dataset to aggregate features, introducing future data to past data. We may assume that in production we would have such information at our disposal.*

In [35]:
trans_count_cli = df[df['target']==0].groupby('client_id_x')['id_trans'].count().reset_index()
trans_count_cli.columns = ['client_id_x', 'trans_count_cli']

df = pd.merge(df, trans_count_cli, on='client_id_x', how='left')

In [47]:
trans_count_cli.sort_values(by='trans_count_cli')

,client_id_x,trans_count_cli
734,1223,460
909,1510,1164
1024,1680,1239
576,955,1291
253,410,1408
...,...,...
551,909,22429
72,114,22433
1082,1776,23263
658,1098,25517


In [36]:
avg_amount_cli = df[df['target']==0].groupby('client_id_x')['amount'].mean().reset_index()
avg_amount_cli.columns = ['client_id_x', 'avg_amount_cli']

df = pd.merge(df, avg_amount_cli, on='client_id_x', how='left')

In [48]:
avg_amount_cli.sort_values(by='avg_amount_cli')

,client_id_x,avg_amount_cli
802,1331,5.466482
811,1348,8.043772
854,1428,8.377451
1181,1942,8.851662
720,1202,11.645583
...,...,...
146,222,131.890845
595,989,134.061426
507,840,140.590702
425,708,145.010430


In [37]:
max_amount_cli = df[df['target']==0].groupby('client_id_x')['amount'].max().reset_index()
max_amount_cli.columns = ['client_id_x', 'max_amount_cli']

df = pd.merge(df, max_amount_cli, on='client_id_x', how='left')

In [49]:
max_amount_cli.sort_values(by='max_amount_cli')

,client_id_x,max_amount_cli
11,19,92.00
1156,1899,99.00
1114,1826,100.00
854,1428,100.00
935,1542,100.00
...,...,...
62,96,4729.38
446,742,5155.36
425,708,5591.73
179,278,5696.78


In [38]:
trans_count_card = df[df['target']==0].groupby('card_id')['id_trans'].count().reset_index()
trans_count_card.columns = ['card_id', 'trans_count_card']

df = pd.merge(df, trans_count_card, on='card_id', how='left')

In [50]:
trans_count_card.sort_values(by='trans_count_card')

,card_id,trans_count_card
3415,5339,2
3770,5781,3
3418,5344,4
264,314,5
1336,2814,10
...,...,...
3597,5570,12886
1683,3239,15967
993,2408,16632
3083,4938,17791


**Observation:** Some clients and cards show extremely high transaction volumes, likely indicating business accounts rather than personal use. In a real-world setting, this should be investigated more carefully.

In [39]:
# Drop irrelevant columns
columns_to_drop = ['id_trans', 'client_id_x', 'card_id', 'merchant_id', 'merchant_state', 'zip', 'mcc', 'id_cli', 'birth_year', 'birth_month', 'address', 'id', 'client_id_y', 'card_number', 'cvv', 'card_on_dark_web', 'Unnamed: 0']
df.drop(columns_to_drop, axis=1, inplace=True)

In [40]:
# Summarize again dataframe

summary_global = column_summary(df)
display(summary_global)

,col_name,col_dtype,num_of_nulls,num_of_non_nulls,num_of_distinct_values,distinct_values_counts
0,date,datetime64[ns],0,8472184,3623402,"{2012-05-20 07:26:00: 13, 2014-12-25 10:40:00:..."
1,amount,float64,0,8472184,70084,"{80.0: 88334, 100.0: 86310, 60.0: 68260, 120.0..."
2,use_chip,object,0,8472184,3,"{'Swipe Transaction': 4407210, 'Chip Transacti..."
3,merchant_city,object,0,8472184,12160,"{'ONLINE': 1041450, 'Houston': 91152, 'Miami':..."
4,errors,object,0,8472184,23,"{'No errors': 8336809, 'Insufficient Balance':..."
5,current_age,int64,0,8472184,74,"{47: 341614, 51: 261452, 52: 250163, 49: 23852..."
6,retirement_age,int64,0,8472184,27,"{65: 1364131, 66: 1350267, 67: 1221393, 68: 84..."
7,gender,object,0,8472184,2,"{'Female': 4343738, 'Male': 4128446}"
8,latitude,float64,0,8472184,740,"{29.76: 113744, 40.84: 72399, 40.64: 67972, 25..."
9,longitude,float64,0,8472184,859,"{-95.38: 124349, -73.94: 67972, -80.2: 63346, ..."


In [41]:
df.head(5)

,date,amount,use_chip,merchant_city,errors,current_age,retirement_age,gender,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,card_brand,card_type,expires,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,merchant_category,target,trans_count_cli,avg_amount_cli,max_amount_cli,trans_count_card
0,2010-01-01 00:02:00,14.57,Swipe Transaction,Bettendorf,No errors,48,67,Male,40.80,-91.12,18076.0,36853.0,112139.0,834,5,Mastercard,Credit,2024-12-01,YES,1,9100.0,2005-09-01,2015,Department Stores,0,7446,33.474038,1139.95,2481
1,2010-01-01 00:02:00,80.00,Swipe Transaction,Vista,No errors,49,65,Male,33.18,-117.29,16894.0,34449.0,36540.0,686,3,Mastercard,Debit,2020-05-01,YES,1,14802.0,2006-01-01,2008,Money Transfer,0,9314,55.776330,1129.61,4283
2,2010-01-01 00:06:00,46.41,Swipe Transaction,Harwood,No errors,51,69,Male,38.86,-76.60,33529.0,68362.0,96182.0,711,2,Visa,Debit,2020-01-01,YES,1,19113.0,2009-07-01,2014,Drinking Places (Alcoholic Beverages),0,4190,80.990702,1244.73,2310
3,2010-01-01 00:07:00,4.81,Swipe Transaction,Bronx,No errors,47,65,Female,40.84,-73.87,25537.0,52065.0,98613.0,828,5,Mastercard,Debit (Prepaid),2014-03-01,YES,1,89.0,2008-01-01,2015,Book Stores,0,12885,51.449454,1744.05,3165
4,2010-01-01 00:14:00,26.46,Online Transaction,ONLINE,No errors,56,65,Male,36.34,-83.28,13668.0,27861.0,108313.0,782,5,Mastercard,Debit (Prepaid),2021-05-01,YES,1,46.0,2007-03-01,2012,Tolls and Bridge Fees,0,9051,32.939269,834.96,2095


## Final Dataset

We have constructed a clean dataset, ready for the modeling phase. This dataset:

- Consolidates all raw tables into a single, unified structure
- Ensures consistent, appropriate data types across all columns
- Removes noise through rare value filtering and null imputation
- Incorporates new features


In [42]:
df.to_csv('../data/CleanData.csv', index=False)